In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

# 1 Load data

## 1.1 Load population

In [2]:
pop_df = pd.read_csv('Pop ML V PIPE.csv', encoding='latin-1', sep='|')
pop_df.head(10)

,MSISDN,STATE_IN,SUBSCRIBER_TYPE_IN,RATE_PLAN,ACCOUNT_ACTIVATED_DATE
0,21697077044,ACTIVE,PREPAID,PRE - Classic,2003-12-19
1,21697076876,ACTIVE,PREPAID,PRE - Classic,2021-11-23
2,21697072547,ACTIVE,PREPAID,PRE - Classic,2020-09-14
3,21697068526,ACTIVE,PREPAID,PRE - Classic,2018-04-11
4,21697061045,ACTIVE,PREPAID,PRE - Classic,2003-11-15
5,21697060361,ACTIVE,PREPAID,PRE - Classic,2003-11-06
6,21697057138,ACTIVE,PREPAID,PRE - Classic,2004-02-01
7,21697046515,ACTIVE,PREPAID,PRE - Classic,2023-04-22
8,21697038087,SUSPENDED,PREPAID,PRE - Classic,2024-03-10
9,21697026893,SUSPENDED,PREPAID,PRE - Classic,2025-01-25


## 1.2 Load SOS

In [3]:
sos_df = pd.read_csv('DS_SOS_ML.csv', encoding='latin-1', sep='|')
sos_df.head(10)

,MSISDN,LOAN_ID,SOS_TYPE,CREDIT_DATE,CREDIT_AMOUNT,TOTAL_REIMBURSED,OUTSTANDING_AMOUNT,LAST_REIMBURSE_DATE,REIMBURSE_RATIO,REIMBURSE_STATUS,DAYS_SINCE_CREDIT
0,21650142555,528139435,CVAS_SOS_CR_DATA,09/01/2026,"4,5","4,5",0,23/01/2026,1,FULL,90
1,21693464197,528150918,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
2,21697349073,528136747,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,10/01/2026,1,FULL,90
3,21698641675,528163694,CVAS_SOS_CR_SOLDE,09/01/2026,3,3,0,15/01/2026,1,FULL,90
4,21694195742,528176161,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,09/01/2026,1,FULL,90
5,21693574965,528132046,CVAS_SOS_CR_SOLDE,09/01/2026,"2,5","2,5",0,19/01/2026,1,FULL,90
6,21696646805,528207682,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,19/01/2026,1,FULL,90
7,21693370117,528155180,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
8,21697385533,528223219,CVAS_SOS_CR_DATA,09/01/2026,"0,25","0,25",0,09/01/2026,1,FULL,90
9,21695867328,528161149,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,10/01/2026,1,FULL,90


# Data interpretation 

In [4]:
pop_df.isnull().sum()

MSISDN                       0
STATE_IN                     0
SUBSCRIBER_TYPE_IN        4442
RATE_PLAN                 4442
ACCOUNT_ACTIVATED_DATE    4442
dtype: int64

In [5]:
sos_df.isnull().sum()

MSISDN                       0
LOAN_ID                      0
SOS_TYPE                     0
CREDIT_DATE                  0
CREDIT_AMOUNT                0
TOTAL_REIMBURSED             0
OUTSTANDING_AMOUNT           0
LAST_REIMBURSE_DATE    2946503
REIMBURSE_RATIO              0
REIMBURSE_STATUS             0
DAYS_SINCE_CREDIT            0
dtype: int64

## Possible  Columns values for population file

In [6]:
for col in pop_df.columns:
    print(f"{col}:")
    print(pop_df[col].unique())
    print("-" * 40)

MSISDN:
[21697077044 21697076876 21697072547 ... 21699004086 21655565721
 21698400893]
----------------------------------------
STATE_IN:
['ACTIVE' 'SUSPENDED' 'ON-HOLD' 'Disconnected Network']
----------------------------------------
SUBSCRIBER_TYPE_IN:
['PREPAID' 'HYB' 'POSTPAID' nan]
----------------------------------------
RATE_PLAN:
['PRE - Classic' 'PRE - Ola' 'PRE - LOL' 'MOBI Plafonné'
 'Corporate Optimum Plus' 'PRE - Club Optimum Plus'
 'PRE - Corporate Optimum' 'PRE - Corporate Optimum Family' 'PRE - Tawwa'
 'Forfait SNJT' 'PRE - Employe TT' 'PRE - TAWWA REINSTALLED'
 'PRE - Double Street' 'PRE - Messenger' 'PRE - Corporate Optimum TT'
 'Forfaits Mobile TT' 'Forfaits Mobile PRO' 'PRE - Trankil TT' 'OPTICA'
 'PRE - 900 bonus' 'PRE - offre 40' 'PRE - Classe_Regions'
 'PRE - Oulidha 1000%' 'PRE - TT 1000%' 'PRE - Day Pass' 'Hayya'
 'Forfait SELECT' 'PRE - Double Reinstal' 'PRE - AHLA' 'PRE - Doublé'
 'PRE - Binetna' 'PRE - Conso 500mil_TT' 'PRE - TT 300%'
 'PRE - E.M. 1000%' 'PR

## Possible Columns values for SOS file

In [7]:
for col in sos_df.columns:
    print(f"{col}:")
    print(sos_df[col].unique())
    print("-" * 40)

MSISDN:
[21650142555 21693464197 21697349073 ... 21697580810 21699605391
 21690563069]
----------------------------------------
LOAN_ID:
[528139435 528150918 528136747 ... 541128113 545729225 544669242]
----------------------------------------
SOS_TYPE:
['CVAS_SOS_CR_DATA' 'CVAS_SOS_CR_SOLDE']
----------------------------------------
CREDIT_DATE:
['09/01/2026' '10/01/2026' '11/01/2026' '12/01/2026' '13/01/2026'
 '14/01/2026' '15/01/2026' '16/01/2026' '17/01/2026' '18/01/2026'
 '19/01/2026' '20/01/2026' '21/01/2026' '22/01/2026' '23/01/2026'
 '24/01/2026' '25/01/2026' '26/01/2026' '27/01/2026' '28/01/2026'
 '29/01/2026' '30/01/2026' '31/01/2026' '01/02/2026' '02/02/2026'
 '03/02/2026' '04/02/2026' '05/02/2026' '06/02/2026' '07/02/2026'
 '08/02/2026' '09/02/2026' '10/02/2026' '11/02/2026' '12/02/2026'
 '13/02/2026' '14/02/2026' '15/02/2026' '16/02/2026' '17/02/2026'
 '18/02/2026' '19/02/2026' '20/02/2026' '21/02/2026' '22/02/2026'
 '23/02/2026' '24/02/2026' '25/02/2026' '26/02/2026' '27/

In [8]:
# Data Preparation for Model

In [9]:
# Merge datasets on MSISDN
df = sos_df.merge(pop_df[['MSISDN', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']], 
                   on='MSISDN', how='inner')

print(f"Merged dataset shape: {df.shape}")
print(f"\nColumns in merged dataset: {df.columns.tolist()}")
print(f"\nNull values after merge:")
print(df[['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']].isnull().sum())

Merged dataset shape: (18562478, 14)

Columns in merged dataset: ['MSISDN', 'LOAN_ID', 'SOS_TYPE', 'CREDIT_DATE', 'CREDIT_AMOUNT', 'TOTAL_REIMBURSED', 'OUTSTANDING_AMOUNT', 'LAST_REIMBURSE_DATE', 'REIMBURSE_RATIO', 'REIMBURSE_STATUS', 'DAYS_SINCE_CREDIT', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']

Null values after merge:
SOS_TYPE                      0
SUBSCRIBER_TYPE_IN        35004
RATE_PLAN                 35004
ACCOUNT_ACTIVATED_DATE    35004
CREDIT_DATE                   0
CREDIT_AMOUNT                 0
dtype: int64


In [10]:
# Data Cleaning - Remove rows with null values in required features
required_cols = ["MSISDN",'SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']
df_clean = df.dropna(subset=required_cols).copy()

print(f"Dataset shape after removing nulls: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
print(f"\nRemaining null values:")
print(df_clean[required_cols].isnull().sum())

Dataset shape after removing nulls: (18527474, 14)
Rows removed: 35004

Remaining null values:
MSISDN                    0
SOS_TYPE                  0
SUBSCRIBER_TYPE_IN        0
ACCOUNT_ACTIVATED_DATE    0
CREDIT_DATE               0
CREDIT_AMOUNT             0
dtype: int64


In [14]:
# Convert date columns to datetime
df_clean['ACCOUNT_ACTIVATED_DATE'] = pd.to_datetime(df_clean['ACCOUNT_ACTIVATED_DATE'], errors='coerce')
df_clean['CREDIT_DATE'] = pd.to_datetime(df_clean['CREDIT_DATE'], errors='coerce')
# Convert amount from text like "4,5" to numeric 4.5
df_clean["CREDIT_AMOUNT"] = (
    df_clean["CREDIT_AMOUNT"]
    .astype(str)
    .str.replace(",", ".", regex=False)
)

df_clean["CREDIT_AMOUNT"] = pd.to_numeric(df_clean["CREDIT_AMOUNT"], errors="coerce")
# Remove rows where date conversion failed
df_clean = df_clean.dropna(subset=['ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE'])

print(f"Dataset shape after date conversion: {df_clean.shape}")
print(f"\nDate columns info:")
print(f"ACCOUNT_ACTIVATED_DATE - min: {df_clean['ACCOUNT_ACTIVATED_DATE'].min()}, max: {df_clean['ACCOUNT_ACTIVATED_DATE'].max()}")
print(f"CREDIT_DATE - min: {df_clean['CREDIT_DATE'].min()}, max: {df_clean['CREDIT_DATE'].max()}")

Dataset shape after date conversion: (7195753, 16)

Date columns info:
ACCOUNT_ACTIVATED_DATE - min: 2000-12-15 00:00:00, max: 2026-04-07 00:00:00
CREDIT_DATE - min: 2026-01-02 00:00:00, max: 2026-12-03 00:00:00


In [15]:
# Feature Engineering - Derive new columns

# CREDIT_DAY_OF_WEEK: Day of week when credit was given (0=Monday, 6=Sunday)
df_clean['CREDIT_DAY_OF_WEEK'] = df_clean['CREDIT_DATE'].dt.dayofweek



# CREDIT_MONTH: Month of the credit date
df_clean['CREDIT_MONTH'] = df_clean['CREDIT_DATE'].dt.month

print("Derived features created:")
print(f"CREDIT_DAY_OF_WEEK - unique values: {sorted(df_clean['CREDIT_DAY_OF_WEEK'].unique())}")
print(f"CREDIT_MONTH - unique values: {sorted(df_clean['CREDIT_MONTH'].unique())}")

print(f"\nData types:")
print(df_clean[['CREDIT_DAY_OF_WEEK', 'CREDIT_MONTH']].dtypes)

Derived features created:
CREDIT_DAY_OF_WEEK - unique values: [0, 1, 2, 3, 4, 5, 6]
CREDIT_MONTH - unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Data types:
CREDIT_DAY_OF_WEEK    int32
CREDIT_MONTH          int32
dtype: object


In [16]:
# User-level features

# Sort by MSISDN and CREDIT_DATE for sequential features
df_clean = df_clean.sort_values(['MSISDN', 'CREDIT_DATE']).reset_index(drop=True)

# 1. user_avg_credit: Average credit amount per user
user_avg_credit = df_clean.groupby('MSISDN')['CREDIT_AMOUNT'].transform('mean')
df_clean['user_avg_credit'] = user_avg_credit

# 2. user_total_loans: Total number of loans per user
user_total_loans = df_clean.groupby('MSISDN')['CREDIT_AMOUNT'].transform('count')
df_clean['user_total_loans'] = user_total_loans

# 3. user_avg_repay_ratio: Average repay ratio (assuming repay_ratio = CREDIT_AMOUNT / user_total_loans)
df_clean['user_avg_repay_ratio'] = df_clean['CREDIT_AMOUNT'] / df_clean['user_total_loans']

# 4. prev_credit_amount: Previous credit amount for each user
df_clean['prev_credit_amount'] = df_clean.groupby('MSISDN')['CREDIT_AMOUNT'].shift(1)
df_clean['prev_credit_amount'].fillna(df_clean['CREDIT_AMOUNT'].median(), inplace=True)

# 5. days_since_last_credit: Days since last credit for each user
df_clean['days_since_last_credit'] = df_clean.groupby('MSISDN')['CREDIT_DATE'].diff().dt.days
df_clean['days_since_last_credit'].fillna(0, inplace=True)

print("User-level features created:")
print(f"  user_avg_credit - shape: {df_clean['user_avg_credit'].shape}")
print(f"  user_total_loans - shape: {df_clean['user_total_loans'].shape}")
print(f"  user_avg_repay_ratio - shape: {df_clean['user_avg_repay_ratio'].shape}")
print(f"  prev_credit_amount - shape: {df_clean['prev_credit_amount'].shape}")
print(f"  days_since_last_credit - shape: {df_clean['days_since_last_credit'].shape}")
print(f"\nDataset shape: {df_clean.shape}")
print(f"\nSample of new features:")
print(df_clean[['MSISDN', 'CREDIT_AMOUNT', 'user_avg_credit', 'user_total_loans', 'user_avg_repay_ratio', 'prev_credit_amount', 'days_since_last_credit']].head(10))

C:\Users\micro\AppData\Local\Temp\ipykernel_14960\1529827527.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_clean['prev_credit_amount'].fillna(df_clean['CREDIT_AMOUNT'].median(), inplace=True)
C:\Users\micro\AppData\Local\Temp\ipykernel_14960\1529827527.py:23: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting 

User-level features created:
  user_avg_credit - shape: (7195753,)
  user_total_loans - shape: (7195753,)
  user_avg_repay_ratio - shape: (7195753,)
  prev_credit_amount - shape: (7195753,)
  days_since_last_credit - shape: (7195753,)

Dataset shape: (7195753, 21)

Sample of new features:
        MSISDN  CREDIT_AMOUNT  user_avg_credit  user_total_loans  \
0  21620000528           3.00           3.0000                 3   
1  21620000528           2.00           3.0000                 3   
2  21620000528           4.00           3.0000                 3   
3  21620001069           2.00           2.0000                 1   
4  21620001196           2.00           1.6250                 4   
5  21620001196           2.00           1.6250                 4   
6  21620001196           2.00           1.6250                 4   
7  21620001196           0.50           1.6250                 4   
8  21620001696           0.25           0.4125                 4   
9  21620001696           0.25 

In [17]:
# Step 1: Analyze cardinality and determine encoding strategy
df_model = df_clean.copy()
df_model['CREDIT_AMOUNT'] = pd.to_numeric(
    df_model['CREDIT_AMOUNT'].replace(',', '.', regex=False),
    errors='coerce'
)
df_model = df_model.dropna(subset=['CREDIT_AMOUNT'])

categorical_cols = [ 'SUBSCRIBER_TYPE_IN' , 'RATE_PLAN', 'SOS_TYPE']
numeric_cols = ['CREDIT_DAY_OF_WEEK', 'CREDIT_MONTH', 'user_avg_credit', 'user_total_loans', 'user_avg_repay_ratio', 'prev_credit_amount', 'days_since_last_credit']


print("=" * 70)
print("CARDINALITY ANALYSIS & ENCODING STRATEGY")
print("=" * 70)

encoder_strategy = {}
for col in categorical_cols:
    n_unique = df_model[col].nunique()
    encoder_type = "OneHotEncoder" if n_unique <= 20 else "OrdinalEncoder"
    encoder_strategy[col] = {
        'unique_values': n_unique,
        'encoder': encoder_type
    }
    print(f"{col}: {n_unique} unique values → {encoder_type}")

print(f"\nTarget variable (CREDIT_AMOUNT):")
print(f"  Min: {df_model['CREDIT_AMOUNT'].min():.2f}")
print(f"  Max: {df_model['CREDIT_AMOUNT'].max():.2f}")
print(f"  Mean: {df_model['CREDIT_AMOUNT'].mean():.2f}")
print(f"\nTotal records: {len(df_model)}")
print("=" * 70)
print(f"  Max: {df_model['CREDIT_AMOUNT'].max():.2f}")
print(f"  Mean: {df_model['CREDIT_AMOUNT'].mean():.2f}")
print(f"\nTotal records: {len(df_model)}")
print("=" * 70)

CARDINALITY ANALYSIS & ENCODING STRATEGY
SUBSCRIBER_TYPE_IN: 3 unique values → OneHotEncoder
RATE_PLAN: 99 unique values → OrdinalEncoder
SOS_TYPE: 2 unique values → OneHotEncoder

Target variable (CREDIT_AMOUNT):
  Min: 0.20
  Max: 55.00
  Mean: 1.56

Total records: 7195753
  Max: 55.00
  Mean: 1.56

Total records: 7195753


In [18]:
# Prepare features and target for modeling (BEFORE train/test split)
# Features: numeric + categorical (NOT yet encoded)
X = df_model[  numeric_cols + categorical_cols].copy()
y = df_model['CREDIT_AMOUNT'].copy()

# Verify data integrity
print(f"Dataset size: {X.shape[0]} rows × {X.shape[1]} features")
print(f"\nNull values in features:")
print(X.isnull().sum())
print(f"\nFeature dtypes:")
print(X.dtypes)
print(f"\nNumeric features summary:")


# Train-Test Split (80-20) - BEFORE fitting any transformers (prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

print(f"\n{'='*70}")
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train/Test ratio: {X_train.shape[0]/X_test.shape[0]:.1f}:1")
print(f"{'='*70}")

Dataset size: 7195753 rows × 10 features

Null values in features:
CREDIT_DAY_OF_WEEK        0
CREDIT_MONTH              0
user_avg_credit           0
user_total_loans          0
user_avg_repay_ratio      0
prev_credit_amount        0
days_since_last_credit    0
SUBSCRIBER_TYPE_IN        0
RATE_PLAN                 0
SOS_TYPE                  0
dtype: int64

Feature dtypes:
CREDIT_DAY_OF_WEEK          int32
CREDIT_MONTH                int32
user_avg_credit           float64
user_total_loans            int64
user_avg_repay_ratio      float64
prev_credit_amount        float64
days_since_last_credit    float64
SUBSCRIBER_TYPE_IN         object
RATE_PLAN                  object
SOS_TYPE                   object
dtype: object

Numeric features summary:

Train set: 5756602 samples
Test set: 1439151 samples
Train/Test ratio: 4.0:1


In [19]:
import numpy as np
from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.metrics import mean_squared_error, r2_score

# -----------------------------
# 1) Preprocessing
# -----------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1,
        dtype=np.float32
    ))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

# Fit preprocessing on train only
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

# -----------------------------
# 2) Transform target manually
# -----------------------------
y_train_log = np.log1p(y_train)

# -----------------------------
# 3) XGBoost model
# -----------------------------
model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse"
)

model.fit(X_train_prepared, y_train_log)

# -----------------------------
# 4) Predictions
# -----------------------------
y_train_pred_log = model.predict(X_train_prepared)
y_test_pred_log = model.predict(X_test_prepared)

# Back to original scale
y_train_pred = np.expm1(y_train_pred_log)
y_test_pred = np.expm1(y_test_pred_log)

# -----------------------------
# 5) Evaluation on original scale
# -----------------------------
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)

train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)

train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 70)
print("XGBOOST REGRESSION MODEL RESULTS")
print("=" * 70)

print("\nTRAINING SET METRICS:")
print(f"  MSE:  {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print("\nTEST SET METRICS:")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# -----------------------------
# 6) Feature importance
# -----------------------------
try:
    feature_names = preprocessor.get_feature_names_out()
    print(f"\nTotal features after preprocessing: {len(feature_names)}")
    print(f"Model type: {type(model).__name__}")

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
        top_idx = np.argsort(importances)[::-1][:10]

        print("\nTop feature importances:")
        for idx in top_idx:
            print(f"  {feature_names[idx]}: {importances[idx]:.6f}")
    else:
        print("\nFeature importances not available.")
except Exception as e:
    print(f"\nCould not extract feature importances: {e}")

XGBOOST REGRESSION MODEL RESULTS

TRAINING SET METRICS:
  MSE:  0.1010
  RMSE: 0.3179
  R²:   0.9761

TEST SET METRICS:
  MSE:  0.0996
  RMSE: 0.3156
  R²:   0.9765

Total features after preprocessing: 10
Model type: XGBRegressor

Top feature importances:
  user_avg_repay_ratio: 0.492579
  user_avg_credit: 0.250403
  user_total_loans: 0.141517
  SOS_TYPE: 0.062231
  days_since_last_credit: 0.019491
  prev_credit_amount: 0.017380
  SUBSCRIBER_TYPE_IN: 0.010247
  RATE_PLAN: 0.003475
  CREDIT_MONTH: 0.002388
  CREDIT_DAY_OF_WEEK: 0.000289
